[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch04-data-engineering/02-nap-du-lieu-tren-cpu.ipynb)

# Nạp dữ liệu trên CPU: thời gian đi đâu, định dạng nào, và tổng hợp ở đâu

Sổ tay 01 tính bằng công thức xem bộ tăng tốc (accelerator) cần bao nhiêu byte mỗi giây. Sổ tay này
đo phía bên kia: CPU của chính máy bạn mất bao lâu để biến một tệp ảnh nén thành một tensor sẵn
sàng cho bước huấn luyện. Rồi nó đo hai lựa chọn mà chương nói quyết định số hạng dữ liệu của định
luật sắt: định dạng tệp và thuật toán nén. Cuối cùng là hai câu hỏi của xử lý phân tán: tổng hợp ở
đâu thì rẻ, và phần nối tiếp chặn tốc độ tới mức nào.

**Bạn sẽ làm:**
1. đo từng khâu của một quy trình nạp ảnh: giải mã, tăng cường dữ liệu, chuẩn hoá, rồi bước huấn
   luyện;
2. tính số tiến trình công nhân (worker process) cần để rót đủ cho bộ tăng tốc, theo phép tính của chương
   và theo tốc độ đo được;
3. chồng việc nạp lên việc tính bằng nạp trước (prefetching) trong cùng tiến trình;
4. ghi một bảng 100 cột ra CSV, JSON và Parquet, rồi đọc 20 cột;
5. so gzip với Snappy;
6. tổng hợp thống kê trên 100 phân vùng, và đọc định luật Amdahl;
7. lưu 10 phiên bản dữ liệu theo nội dung thay vì chép nguyên bản.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import hashlib
import io
import queue
import shutil
import threading

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
dat_hat_giong()


def bam_gio(ham, lap=3):
    """Thời gian ngắn nhất trong `lap` lần chạy: lần chậm thường là do máy đang bận việc khác."""
    tot = float("inf")
    for _ in range(lap):
        t0 = time.perf_counter()
        ham()
        tot = min(tot, time.perf_counter() - t0)
    return tot


print("sẵn sàng")

## 1. Một mẫu đi qua quy trình nạp tốn bao nhiêu

Kho dữ liệu ảnh thật lưu ảnh ở dạng nén, PNG hoặc JPEG. Mỗi lần huấn luyện đọc một ảnh, CPU phải
giải mã nó thành mảng điểm ảnh, tăng cường dữ liệu (data augmentation) bằng vài phép biến đổi ngẫu
nhiên, rồi chuẩn hoá thành tensor. Sổ tay dựng hai kho từ 4,000 ảnh MNIST, mã hoá một lần trong bộ
nhớ:

* **kho nhỏ:** ảnh xám 28×28 lưu dạng PNG, đúng cỡ của MNIST;
* **kho lớn:** 640 ảnh phóng lên 224×224, ba kênh màu, lưu dạng JPEG, cỡ ảnh đầu vào quen thuộc của
  ResNet-50.

Chữ số trên nền đen nén tốt hơn ảnh chụp nhiều, nên tệp JPEG ở đây nhỏ hơn ảnh thật, và giải mã có
lẽ nhanh hơn ảnh thật cùng kích thước. Hãy coi kho lớn là một cận dưới của chi phí.

In [ ]:
tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
ANH = tap.data[:4000].numpy()
NHAN = tap.targets[:4000].numpy()


def ma_hoa(a, kieu, canh=None):
    im = Image.fromarray(a)
    if canh:
        im = im.resize((canh, canh), Image.BILINEAR).convert("RGB")
    b = io.BytesIO()
    im.save(b, format=kieu, **({"quality": 90} if kieu == "JPEG" else {}))
    return b.getvalue()


KHO = {"nhỏ: PNG 28×28": ([ma_hoa(a, "PNG") for a in ANH], "L", 64),
       "lớn: JPEG 224×224": ([ma_hoa(a, "JPEG", 224) for a in ANH[:640]], "RGB", 32)}
for ten, (kho, _, _) in KHO.items():
    print(f"kho {ten}: {len(kho):,} tệp, trung bình {np.mean([len(b) for b in kho]) / 1e3:.2f} KB một tệp")

Bốn khâu, mỗi khâu một hàm. Tăng cường ở đây là xoay nhẹ và dịch ngẫu nhiên (đệm viền rồi cắt lại),
không lật ngang, vì một chữ số lật ngang không còn là chữ số đó. Mỗi kho có một mạng tích chập nhỏ
riêng để đo bước huấn luyện. Thời gian của bước huấn luyện được chia đều cho các mẫu trong một batch.

**Dự đoán:** với ảnh 28×28, khâu nào tốn nhất? Còn với ảnh 224×224 thì sao?

In [ ]:
def giai_ma(b, mode):
    return np.asarray(Image.open(io.BytesIO(b)).convert(mode))


def tang_cuong(a, rng):
    """Xoay trong ±10 độ, rồi dịch tối đa 1/14 cạnh ảnh theo mỗi chiều."""
    a = np.asarray(Image.fromarray(a).rotate(float(rng.uniform(-10, 10)), resample=Image.BILINEAR))
    h = a.shape[0]
    p = h // 14
    a = np.pad(a, ((p, p), (p, p)) + (((0, 0),) if a.ndim == 3 else ()))
    i, j = rng.integers(0, 2 * p + 1, 2)
    return a[i:i + h, j:j + h]


def chuan_hoa(a):
    x = (torch.from_numpy(np.array(a, dtype=np.float32)) / 255 - 0.5) / 0.5
    return x.permute(2, 0, 1) if x.ndim == 3 else x[None]


def tao_mang(mode):
    torch.manual_seed(0)
    if mode == "L":
        return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                             nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                             nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
    return nn.Sequential(nn.Conv2d(3, 8, 5, stride=4, padding=2), nn.ReLU(),
                         nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10))


def buoc_huan_luyen(mang, opt):
    def buoc(x, y):
        opt.zero_grad()
        F.cross_entropy(mang(x), y).backward()
        opt.step()
    return buoc


def do_tung_khau(kho, mode, lo, n=320):
    rng = np.random.default_rng(0)
    mau = kho[:n]
    anh = [giai_ma(b, mode) for b in mau]
    sau = [tang_cuong(a, rng) for a in anh]
    x = torch.stack([chuan_hoa(a) for a in sau[:lo]])
    y = torch.as_tensor(NHAN[:lo])
    mang = tao_mang(mode)
    buoc = buoc_huan_luyen(mang, torch.optim.SGD(mang.parameters(), lr=0.05))
    buoc(x, y)                                  # lần đầu chậm hơn: không tính
    return {"giải mã": bam_gio(lambda: [giai_ma(b, mode) for b in mau]) / n,
            "tăng cường": bam_gio(lambda: [tang_cuong(a, rng) for a in anh]) / n,
            "chuẩn hoá": bam_gio(lambda: [chuan_hoa(a) for a in sau]) / n,
            "bước huấn luyện": bam_gio(lambda: [buoc(x, y) for _ in range(4)]) / (4 * lo)}


KHAU = {}
for ten, (kho, mode, lo) in KHO.items():
    KHAU[ten] = do_tung_khau(kho, mode, lo)
    for khau, giay in KHAU[ten].items():
        do_tren_may(f"kho {ten}, {khau}", round(giay * 1e6, 1), "µs mỗi mẫu")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for y, (ten, khau) in enumerate(KHAU.items()):
    tong = sum(khau.values())
    trai = 0.0
    for (k, giay), mau in zip(khau.items(), MAU):
        phan = 100 * giay / tong
        ax.barh(y, phan, left=trai, color=mau, height=0.55, edgecolor="white", lw=2,
                label=k if y == 0 else None)
        if phan > 7:
            ax.text(trai + phan / 2, y, f"{phan:.0f} %", ha="center", va="center", fontsize=8, color="white")
        trai += phan
    ax.text(101.5, y, f"{tong * 1e6:,.0f} µs\nmỗi mẫu", va="center", fontsize=8, color="0.2")
ax.set_yticks(range(len(KHAU)), [f"kho {t}" for t in KHAU], fontsize=9)
ax.set_xlim(0, 118)
ax.set_xticks(range(0, 101, 20))
ax.set_xlabel("phần thời gian của một mẫu (%)")
ax.set_title("Thời gian của một mẫu đi đâu (đo trên máy này)")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.3), ncol=4, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
so_diem = 224 * 224 * 3 / (28 * 28)
print(f"ảnh lớn có {so_diem:.0f} lần số giá trị điểm ảnh của ảnh nhỏ")
for k in ("giải mã", "tăng cường", "chuẩn hoá"):
    do_tren_may(f"{k}: ảnh lớn so với ảnh nhỏ", round(KHAU["lớn: JPEG 224×224"][k] / KHAU["nhỏ: PNG 28×28"][k], 1), "lần")

Ba khâu chuẩn bị đều đắt lên khi ảnh lớn hơn, nhưng thường không tăng đủ 192 lần, vì mỗi lời gọi còn
một phần phí cố định không phụ thuộc kích thước ảnh. Chương nói đúng điều này về thị giác máy tính:
giải mã JPEG độ phân giải cao và tăng cường dữ liệu trên CPU có thể chiếm phần lớn đường vào. Bước
huấn luyện ở đây là một mạng rất nhỏ chạy trên CPU. Với ResNet-50 trên bộ tăng tốc của napkin math
1.9, trần compute là 12,682 ảnh mỗi giây, tức chưa tới 0.1 ms mỗi ảnh, nên phần chuẩn bị trên CPU càng
dễ là điểm nghẽn.

## 2. Cần bao nhiêu tiến trình công nhân

Một tiến trình công nhân làm tuần tự cả ba khâu chuẩn bị cho từng ảnh. Muốn rót đủ cho bộ tăng tốc,
số công nhân phải đủ để tổng tốc độ của họ đuổi kịp tốc độ tiêu thụ. Chương làm một phép tính minh
hoạ: mỗi công nhân cấp 375 ảnh mỗi giây, bộ tăng tốc tiêu thụ 3,000 ảnh mỗi giây.

In [ ]:
R_CONG_NHAN = sach(375, trich="375 images/s")
R_TIEU_THU = sach(3000, trich="3,000 images/s")
can = math.ceil(R_TIEU_THU / R_CONG_NHAN)
theo_sach("số công nhân để tới điểm giao", can, sach=8, trich="8 workers are needed")

r_may = 1 / sum(v for k, v in KHAU["lớn: JPEG 224×224"].items() if k != "bước huấn luyện")
do_tren_may("một lõi của máy này, kho lớn", round(r_may), "ảnh/giây")
TRAN_19 = sach(12_682, nguon="24,6 GFLOP/ảnh · 12.682 img/s")   # trần compute của napkin math 1.9
for muc_tieu in (R_TIEU_THU, TRAN_19):
    do_tren_may(f"số lõi như thế để đạt {muc_tieu:,} ảnh/giây", math.ceil(muc_tieu / r_may), "lõi")

so_cn = np.arange(1, 17)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(so_cn, np.minimum(so_cn * R_CONG_NHAN, R_TIEU_THU), "o-", color=MAU[0], lw=2, ms=5,
        label="phép tính minh hoạ của chương: 375 ảnh/giây mỗi công nhân")
ax.plot(so_cn, np.minimum(so_cn * r_may, R_TIEU_THU), "s--", color=MAU[1], lw=1.5, ms=5,
        label="mỗi công nhân bằng một lõi của máy này (đo trên máy này)")
ax.axhline(R_TIEU_THU, color="0.4", lw=1, ls=":")
ax.text(16.4, R_TIEU_THU * 1.02, "bộ tăng tốc:\n3,000 ảnh/giây", fontsize=8, va="bottom", color="0.25")
ax.annotate(f"{can} công nhân", (can, R_TIEU_THU), xytext=(can + 0.6, R_TIEU_THU * 0.55),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.7))
ax.set_xlim(0.5, 19.5)
ax.set_xticks(range(1, 17))
ax.set_ylim(0, R_TIEU_THU * 1.25)
ax.set_xlabel("số tiến trình công nhân")
ax.set_ylabel("ảnh mỗi giây tới được bộ tăng tốc")
ax.set_title("Dưới điểm giao, quy trình nạp chặn; trên điểm giao, bộ tăng tốc chặn")
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Chương nhấn mạnh 8 là một phép khớp năng lực, không phải hằng số phần cứng: số đó đổi theo phép biến
đổi ảnh, tầng lưu trữ, kích thước batch và CPU của máy chủ. Đường đo của bạn là một ví dụ: nó giả định
mỗi lõi thêm vào cho đúng tốc độ của lõi đầu, mà thực tế các lõi còn chia nhau băng thông bộ nhớ.
Colab chỉ có 2 lõi, nên sổ tay giữ num_workers = 0 và tính phần còn lại bằng công thức.

## 3. Nạp trước: chồng việc nạp lên việc tính

Bộ nạp dữ liệu (data loader) của các framework dùng hàng đợi đệm kép: trong khi bộ tăng tốc tính batch
$k$, CPU đã chuẩn bị batch $k+1$. Nạp trước không làm khâu nào nhanh hơn; nó cho hai khâu chạy cùng
lúc. Nếu một batch tốn $T_\text{nạp}$ để chuẩn bị và $T_\text{tính}$ để huấn luyện, thì

$$\max(T_\text{nạp}, T_\text{tính}) \;\le\; T_\text{mỗi batch} \;\le\; T_\text{nạp} + T_\text{tính}.$$

Cận dưới là khi chồng hoàn toàn, cận trên là khi chạy nối tiếp. Sổ tay chạy việc nạp trên một luồng
nền với hàng đợi hai chỗ, vẫn trong cùng tiến trình.

**Dự đoán:** với kho nào thì nạp trước rút ngắn thời gian nhiều hơn?

In [ ]:
def nap_lo(kho, mode, chi_so, rng):
    x = torch.stack([chuan_hoa(tang_cuong(giai_ma(kho[i], mode), rng)) for i in chi_so])
    return x, torch.as_tensor(NHAN[chi_so])


def chay(kho, mode, lo, cach):
    """cach: 'chỉ nạp', 'chỉ tính', 'nối tiếp' hoặc 'nạp trước'. Trả về số giây cho cả lượt."""
    rng = np.random.default_rng(1)
    thu_tu = np.random.default_rng(2).permutation(len(kho))
    cac_lo = [thu_tu[k * lo:(k + 1) * lo] for k in range(len(kho) // lo)]
    mang = tao_mang(mode)
    buoc = buoc_huan_luyen(mang, torch.optim.SGD(mang.parameters(), lr=0.05))
    lo_mau = nap_lo(kho, mode, cac_lo[0], np.random.default_rng(3))
    buoc(*lo_mau)                               # khởi động, không tính
    t0 = time.perf_counter()
    if cach == "chỉ nạp":
        for cs in cac_lo:
            nap_lo(kho, mode, cs, rng)
    elif cach == "chỉ tính":
        for _ in cac_lo:
            buoc(*lo_mau)
    elif cach == "nối tiếp":
        for cs in cac_lo:
            buoc(*nap_lo(kho, mode, cs, rng))
    else:
        hang = queue.Queue(maxsize=2)           # đệm kép

        def nap():
            for cs in cac_lo:
                hang.put(nap_lo(kho, mode, cs, rng))
            hang.put(None)

        luong = threading.Thread(target=nap, daemon=True)
        luong.start()
        while (lo_ke := hang.get()) is not None:
            buoc(*lo_ke)
        luong.join()
    return time.perf_counter() - t0


CACH = ("chỉ nạp", "chỉ tính", "nối tiếp", "nạp trước")
DO_NAP = {}
for ten, (kho, mode, lo) in KHO.items():
    DO_NAP[ten] = {c: min(chay(kho, mode, lo, c) for _ in range(2)) for c in CACH}
    for c in CACH:
        do_tren_may(f"kho {ten}, {c}", round(DO_NAP[ten][c], 2), "giây cho cả lượt")
    d = DO_NAP[ten]
    do_tren_may(f"kho {ten}, nạp trước tiết kiệm", round(100 * (1 - d["nạp trước"] / d["nối tiếp"])), "%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (ten, d) in zip(axes, DO_NAP.items()):
    ax.bar(range(4), [d[c] for c in CACH], color=[MAU[0], MAU[1], "0.55", MAU[2]], width=0.6,
           edgecolor="white", lw=2)
    ax.axhline(max(d["chỉ nạp"], d["chỉ tính"]), color="0.3", lw=1, ls=":")
    ax.axhline(d["chỉ nạp"] + d["chỉ tính"], color="0.3", lw=1, ls="--")
    ax.text(3.4, max(d["chỉ nạp"], d["chỉ tính"]), "cận dưới\n(max)", fontsize=7, va="bottom")
    ax.text(3.4, d["chỉ nạp"] + d["chỉ tính"], "cận trên\n(tổng)", fontsize=7, va="bottom")
    ax.set_xticks(range(4), CACH, fontsize=8)
    ax.set_xlim(-0.6, 4.3)
    ax.set_ylim(0, (d["chỉ nạp"] + d["chỉ tính"]) * 1.2)
    ax.set_title(f"kho {ten}", fontsize=9)
axes[0].set_ylabel("giây cho một lượt qua kho")
fig.suptitle("Nạp trước trên một luồng nền (đo trên máy này)", fontsize=11)
plt.tight_layout()
plt.show()

Lợi ích lớn nhất khi hai khâu dài gần bằng nhau, vì khi đó một nửa thời gian nối tiếp được giấu
đi. Khi một khâu dài hơn hẳn khâu kia, nạp trước chỉ giấu được khâu ngắn, và tổng thời gian về gần
khâu dài. Trong Python, hai luồng còn chia nhau khoá GIL ở những đoạn mã thuần Python, nên kết quả
đo có thể nằm giữa hai cận thay vì chạm cận dưới. Đó là một lý do bộ nạp dữ liệu của framework dùng
tiến trình công nhân riêng chứ không dùng luồng.

## 4. Định dạng tệp: đọc 20 cột trong 100 cột

Napkin math 1.10 định nghĩa hiệu suất định dạng $\eta_\text{format}$ là phần byte đọc vào thực sự
mang đặc trưng được chọn, và phương trình 7 dùng nó để nhân băng thông vật lý:

$$\text{băng thông hiệu dụng} = \text{băng thông vật lý} \times \eta_\text{format}.$$

Kịch bản của chương: một mô hình phát hiện gian lận dùng 20 đặc trưng trong một bảng 100 cột.

In [ ]:
COT_CAN = sach(20, nguon="20 trong 100 cột · η = 0,2")
COT_TONG = sach(100, nguon="20 trong 100 cột · η = 0,2")
eta_hang = COT_CAN / COT_TONG
theo_sach("η của định dạng theo hàng (CSV)", eta_hang, sach=0.2, nguon="20 trong 100 cột · η = 0,2")
theo_sach("băng thông đĩa bị bỏ phí (%)", 100 * (1 - eta_hang), sach=80, nguon="5× · bỏ 80 % băng thông")
ETA_COT = sach(1, nguon="η = 0,2 vs ~1 · 5×", tol=0.05)       # theo cột, bỏ qua phần siêu dữ liệu
theo_sach("thông lượng hiệu dụng, theo cột so với theo hàng", ETA_COT / eta_hang, sach=5,
          nguon="5× · bỏ 80 % băng thông", tol=0.25)

Giờ dựng một bảng như thế. 10,000 giao dịch, 100 cột: 60 cột số thực, 25 cột đếm, 10 cột phân loại
mỗi cột 200 giá trị, và 5 cột cờ đúng sai. Các phân phối là **giả định** của sổ tay. Bảng được ghi ra
bốn dạng: CSV và JSON (theo hàng, dạng văn bản), Parquet với hai cách nén (định dạng theo cột, hay
columnar format, dạng nhị phân).
Các tệp ghi vào một thư mục tạm bên cạnh sổ tay và bị xoá ở cuối phần này.

**Dự đoán:** khi chỉ cần 20 cột, đọc CSV có nhanh hơn rõ rệt so với đọc cả 100 cột không?

In [ ]:
rng = np.random.default_rng(4)
N_DONG = 10_000
cot = {}
for i in range(60):
    cot[f"so_{i:02d}"] = np.round(rng.lognormal(rng.uniform(0, 3), 0.8, N_DONG), 2)
for i in range(25):
    cot[f"dem_{i:02d}"] = rng.poisson(rng.uniform(1, 50), N_DONG)
NHOM = np.array([f"nhom_{k:03d}" for k in range(200)])
P_NHOM = 1 / np.arange(1, 201)
for i in range(10):
    cot[f"loai_{i:02d}"] = rng.choice(NHOM, N_DONG, p=P_NHOM / P_NHOM.sum())
for i in range(5):
    cot[f"co_{i}"] = rng.random(N_DONG) < 0.1
bang = pd.DataFrame(cot)
CHON = [f"so_{i:02d}" for i in range(12)] + [f"dem_{i:02d}" for i in range(4)] + ["loai_00", "loai_01", "co_0", "co_1"]
assert bang.shape == (N_DONG, COT_TONG) and len(CHON) == COT_CAN

TAM = "bang_tam"
os.makedirs(TAM, exist_ok=True)
DANG = {"CSV": ("bang.csv", lambda p: bang.to_csv(p, index=False),
                lambda p: pd.read_csv(p), lambda p: pd.read_csv(p, usecols=CHON)),
        "JSON theo dòng": ("bang.jsonl", lambda p: bang.to_json(p, orient="records", lines=True),
                           lambda p: pd.read_json(p, lines=True), lambda p: pd.read_json(p, lines=True)[CHON]),
        "Parquet, Snappy": ("bang_snappy.parquet", lambda p: bang.to_parquet(p, compression="snappy"),
                            lambda p: pd.read_parquet(p), lambda p: pd.read_parquet(p, columns=CHON)),
        "Parquet, gzip": ("bang_gzip.parquet", lambda p: bang.to_parquet(p, compression="gzip"),
                          lambda p: pd.read_parquet(p), lambda p: pd.read_parquet(p, columns=CHON))}
KQ = {}
for ten, (tep, ghi, doc_het, doc_chon) in DANG.items():
    p = os.path.join(TAM, tep)
    ghi(p)
    KQ[ten] = {"MB": os.path.getsize(p) / 1e6, "đọc 100 cột": bam_gio(lambda: doc_het(p), 2),
               "đọc 20 cột": bam_gio(lambda: doc_chon(p), 2)}
    print(f"{ten:>16}: {KQ[ten]['MB']:.2f} MB trên đĩa")
    do_tren_may(f"{ten}, đọc 100 cột", round(KQ[ten]["đọc 100 cột"] * 1e3, 1), "ms")
    do_tren_may(f"{ten}, đọc 20 cột", round(KQ[ten]["đọc 20 cột"] * 1e3, 1), "ms")

Bây giờ đếm byte, không đếm giây. Với CSV, lấy 20 cột vẫn phải đọc và tách cả dòng, nên phần byte
hữu ích là độ dài của 20 trường chia cho độ dài cả dòng. Với Parquet, siêu dữ liệu ở cuối tệp ghi
từng khối cột nằm ở đâu và dài bao nhiêu, nên đọc 20 cột chỉ chạm tới 20 khối cột cộng phần siêu dữ
liệu.

In [ ]:
p_csv = os.path.join(TAM, "bang.csv")
eta_csv = len(bang[CHON].to_csv(index=False)) / len(bang.to_csv(index=False))
md = pq.ParquetFile(os.path.join(TAM, "bang_snappy.parquet")).metadata
byte_chon = sum(md.row_group(g).column(c).total_compressed_size
                for g in range(md.num_row_groups) for c in range(md.num_columns)
                if md.row_group(g).column(c).path_in_schema in CHON)
byte_cot = sum(md.row_group(g).column(c).total_compressed_size
               for g in range(md.num_row_groups) for c in range(md.num_columns))
print(f"CSV: phải đọc 100 % tệp, trong đó {100 * eta_csv:.1f} % là 20 trường cần dùng (η ≈ {eta_csv:.2f})")
print(f"Parquet: đọc {100 * byte_chon / byte_cot:.1f} % số byte dữ liệu của tệp, cộng {md.serialized_size:,} byte "
      f"siêu dữ liệu, nên η ≈ {byte_chon / (byte_chon + md.serialized_size):.2f}")

ten_dang = list(KQ)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].barh(range(4), [KQ[t]["MB"] for t in ten_dang], color=MAU[0], height=0.55)
axes[0].set_yticks(range(4), ten_dang, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("MB trên đĩa")
axes[0].set_title("Kích thước tệp", fontsize=9)
for y, t in enumerate(ten_dang):
    het, chon = KQ[t]["đọc 100 cột"] * 1e3, KQ[t]["đọc 20 cột"] * 1e3
    axes[1].plot([chon, het], [y, y], color="0.75", lw=2, zorder=1)
    axes[1].plot(het, y, "o", color=MAU[1], ms=8, mec="white", mew=1.5, zorder=2, label="đọc 100 cột" if y == 0 else None)
    axes[1].plot(chon, y, "s", color=MAU[2], ms=8, mec="white", mew=1.5, zorder=3, label="đọc 20 cột" if y == 0 else None)
    axes[1].text(max(het, chon) * 1.25, y, f"20 cột: {chon:.0f} ms\n100 cột: {het:.0f} ms", va="center", fontsize=7, color="0.3")
axes[1].set_xscale("log")
axes[1].set_xlim(1, 3000)
axes[1].set_ylim(3.6, -0.6)
axes[1].set_yticks(range(4), [""] * 4)
axes[1].set_xlabel("ms (thang log)")
axes[1].set_title("Thời gian đọc (đo trên máy này)", fontsize=9)
axes[1].legend(loc="lower right", fontsize=8, frameon=True, framealpha=0.9, edgecolor="none")
plt.tight_layout()
plt.show()

Kích thước tệp và phần byte là phép đếm, nên ra như nhau trên mọi máy chạy cùng phiên bản thư viện.
Với Parquet, η không chạm 1 vì phần siêu dữ liệu: trên một tệp nhỏ cỡ vài MB như ở đây, nó không nhỏ
chút nào, còn trên một tệp hàng GB thì gần như biến mất. Đó là điều chương nói khi viết "bỏ qua phần
siêu dữ liệu".
Thời gian đọc thì đo, và khoảng cách giữa các định dạng còn gồm cả chi phí phân tích văn bản (parse)
mà η không tính: CSV và JSON phải đổi từng ký tự thành số, còn Parquet đọc thẳng số nhị phân. Chương
xếp điều này vào các ngộ nhận: tối ưu kỹ kernel của bộ tăng tốc mà vẫn để dữ liệu nạp ở dạng JSON
hay CSV. Lời khuyên của chương là đo các định dạng trên đúng kiểu truy cập của mình rồi mới chuyển.

## 5. Nén: tỉ lệ nén hay tốc độ giải nén

Chương so hai thuật toán nén: gzip nén chặt hơn, Snappy giải nén nhanh hơn. Với một tập 100 GB đọc
qua 50 epoch, chênh lệch tốc độ giải nén cộng dồn thành hàng giờ.

In [ ]:
GIAI_NEN = {"gzip": sach(120, trich="gzip’s 120 MB/s") * 1e6,
            "Snappy": sach(500, nguon="gzip 6–8× vs Snappy 2–3× · 500 MB/s") * 1e6}
theo_sach("Snappy giải nén nhanh hơn gzip", GIAI_NEN["Snappy"] / GIAI_NEN["gzip"], sach=4.2,
          nguon="500 MB/s · 4,2× nhanh hơn")
TAP_GB = sach(100, trich="100 GB dataset compressed with gzip")
phut = {k: TAP_GB * 1e9 / v / 60 for k, v in GIAI_NEN.items()}
theo_sach("gzip, phút giải nén mỗi epoch", phut["gzip"], sach=13.9, trich="13.9 minutes")
theo_sach("Snappy, phút giải nén mỗi epoch", phut["Snappy"], sach=3.3, trich="3.3 minutes")
theo_sach("chênh lệch mỗi epoch (phút)", phut["gzip"] - phut["Snappy"], sach=10.6, trich="10.6 minutes difference")
EPOCH = sach(50, trich="50 epochs")
theo_sach("cộng dồn qua 50 epoch (giờ)", (phut["gzip"] - phut["Snappy"]) * EPOCH / 60, sach=9,
          trich="compounds to 9 hours", tol=0.5)

Con số đó là 8.8 giờ, chương làm tròn thành 9. Bây giờ đo hai thuật toán trên máy của bạn, trên
chính tệp CSV vừa ghi, mỗi thuật toán một luồng. Tỉ lệ nén phụ thuộc rất mạnh vào dữ liệu: văn bản
lặp nhiều thì nén tốt, số ngẫu nhiên thì gần như không nén được. Bảng giả định ở đây không phải dữ
liệu của chương, nên đừng so tỉ lệ đo được với 6–8 lần và 2–3 lần của chương.

**Dự đoán:** trên CSV này, gzip nén chặt hơn Snappy bao nhiêu lần, và đổi lại chậm hơn bao nhiêu?

In [ ]:
tho = open(p_csv, "rb").read()
NEN = {}
for ten, codec in (("gzip", "gzip"), ("Snappy", "snappy")):
    nen = pa.compress(tho, codec=codec, asbytes=True)
    giay = bam_gio(lambda: pa.decompress(nen, decompressed_size=len(tho), codec=codec, asbytes=True))
    NEN[ten] = {"tỉ lệ": len(tho) / len(nen), "MB/s": len(tho) / giay / 1e6}
    print(f"{ten}: nén {len(tho) / 1e6:.2f} MB xuống {len(nen) / 1e6:.2f} MB, tỉ lệ {NEN[ten]['tỉ lệ']:.2f} lần")
    do_tren_may(f"{ten}, tốc độ giải nén", round(NEN[ten]["MB/s"]), "MB/s")
gio_50 = {k: TAP_GB * 1e9 / (v["MB/s"] * 1e6) * EPOCH / 3600 for k, v in NEN.items()}
for k, v in gio_50.items():
    do_tren_may(f"{k}: giải nén 100 GB qua 50 epoch ở tốc độ đo được", round(v, 2), "giờ")
shutil.rmtree(TAM)                              # dọn thư mục tạm

Nén chặt hơn tiết kiệm tiền lưu trữ và băng thông đọc; giải nén nhanh hơn tiết kiệm thời gian CPU ở
mỗi epoch. Chương kết luận rằng với huấn luyện, nơi thông lượng (throughput) quan trọng hơn chi phí
lưu trữ, tốc độ của Snappy thường thắng phần tiết kiệm dung lượng của gzip. Chữ "thường" quan trọng:
nếu đĩa hay mạng mới là điểm nghẽn, mỗi byte nén bớt là một byte không phải truyền, và cán cân có thể
nghiêng về phía nén chặt.

## 6. Tổng hợp ở đâu: thuế phối hợp và định luật Amdahl

Napkin math 1.7: cần tính trung bình của 1 TB đặc trưng nằm rải trên 100 nút để chuẩn hoá. Cách A gom
hết về một nút rồi tính. Cách B cho mỗi nút tính phần của mình rồi gửi kết quả nhỏ về.

**Dự đoán:** cách B nhanh hơn cách A bao nhiêu lần: vài lần, vài chục lần, hay hàng trăm lần?

In [ ]:
DL_TB = sach(1, nguon="1 TB trên 100 nút · 10 GB/s · 100 giây")
SO_NUT = sach(100, nguon="1 TB trên 100 nút · 10 GB/s · 100 giây")
MANG = sach(10, nguon="1 TB trên 100 nút · 10 GB/s · 100 giây") * 1e9
RAM = (sach(50, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM") * 1e9,
       sach(200, nguon="100–200 MB/s đĩa quay vs 50–200 GB/s RAM") * 1e9)
truyen = DL_TB * 1e12 / MANG
theo_sach("cách A: truyền 1 TB (giây)", truyen, sach=100, nguon="1 TB trên 100 nút · 10 GB/s · 100 giây")
tinh_a = [DL_TB * 1e12 / bw for bw in RAM[::-1]]
theo_sach("cách A: tính trên một nút, nhanh nhất (giây)", tinh_a[0], sach=5, trich="5–20 seconds at RAM bandwidth")
theo_sach("cách A: tính trên một nút, chậm nhất (giây)", tinh_a[1], sach=20, trich="5–20 seconds at RAM bandwidth")
theo_sach("cách A: tổng, nhanh nhất (giây)", truyen + tinh_a[0], sach=105, nguon="105–120 giây vs 0,05–0,2 giây")
theo_sach("cách A: tổng, chậm nhất (giây)", truyen + tinh_a[1], sach=120, nguon="105–120 giây vs 0,05–0,2 giây")
moi_nut = DL_TB * 1e12 / SO_NUT
tinh_b = [moi_nut / bw for bw in RAM[::-1]]
theo_sach("cách B: mỗi nút tính phần mình, nhanh nhất (giây)", tinh_b[0], sach=0.05,
          trich="0.05–0.2 seconds (10 GB at RAM speed)")
theo_sach("cách B: mỗi nút tính phần mình, chậm nhất (giây)", tinh_b[1], sach=0.2,
          trich="0.05–0.2 seconds (10 GB at RAM speed)")
BYTE_GUI = sach(8, trich="8 bytes each")
print(f"cách B gửi {SO_NUT} × {BYTE_GUI} byte = {SO_NUT * BYTE_GUI} byte qua mạng, "
      f"tức {SO_NUT * BYTE_GUI / MANG * 1e9:.0f} ns ở 10 GB/s, chưa tính độ trễ")
print(f"cách B nhanh hơn cách A từ {(truyen + tinh_a[0]) / tinh_b[1]:,.0f} tới {(truyen + tinh_a[1]) / tinh_b[0]:,.0f} lần")
SSD = sach(5, nguon="256 GB RAM · 5 GB/s SSD · 200 giây") * 1e9
theo_sach("đọc 1 TB từ SSD 5 GB/s (giây)", DL_TB * 1e12 / SSD, sach=200, nguon="256 GB RAM · 5 GB/s SSD · 200 giây")

Truyền qua mạng 10 GB/s và đọc từ SSD 5 GB/s cùng một bậc độ lớn, trong khi RAM nhanh hơn cả hai một
bậc. Đó là lý do chương nói compute đi theo dữ liệu. Có một điều kiện: chương nói cách B gửi các trung
bình riêng phần, nhưng phần nhận xét của nó nói rõ một phép trung bình phân tán phải mang theo **tổng
và số đếm**. Lý do: nếu các phân vùng không bằng nhau, trung bình của các trung bình không phải trung
bình chung. Hãy thử trên 100 phân vùng có kích thước khác nhau.

In [ ]:
rng = np.random.default_rng(5)
co = rng.integers(1_000, 30_000, SO_NUT)                    # số dòng mỗi phân vùng: giả định
phan_vung = [rng.normal(5 + 10 * n / 30_000, 2, n) for n in co]   # phân vùng đông có giá trị lớn hơn
toan_bo = np.concatenate(phan_vung)

# mỗi nút gửi về (số đếm, tổng, tổng bình phương độ lệch quanh trung bình riêng)
rieng = [(len(v), v.sum(), ((v - v.mean()) ** 2).sum()) for v in phan_vung]


def gop(a, b):
    """Gộp hai bộ (n, tổng, M2): công thức gộp phương sai theo cặp."""
    na, sa, ma = a
    nb, sb, mb = b
    d = sb / nb - sa / na
    return na + nb, sa + sb, ma + mb + d * d * na * nb / (na + nb)


n, s, m2 = rieng[0]
for r in rieng[1:]:
    n, s, m2 = gop((n, s, m2), r)
assert np.isclose(s / n, toan_bo.mean()) and np.isclose(m2 / n, toan_bo.var())
tb_cua_tb = np.mean([v.mean() for v in phan_vung])
print(f"trung bình chung, tính trên cả {len(toan_bo):,} dòng: {toan_bo.mean():.4f}")
print(f"gộp (số đếm, tổng):                  {s / n:.4f}  (khớp)")
print(f"trung bình của 100 trung bình:       {tb_cua_tb:.4f}  (lệch {tb_cua_tb - toan_bo.mean():+.4f})")
print(f"phương sai gộp từ (n, tổng, M2):     {m2 / n:.4f}  (khớp với {toan_bo.var():.4f})")
trung_vi_cua_tv = np.median([np.median(v) for v in phan_vung])
print(f"trung vị chung {np.median(toan_bo):.4f}, trung vị của các trung vị {trung_vi_cua_tv:.4f}: "
      f"không có bộ tóm tắt cỡ cố định nào gộp ra trung vị đúng")

Trung bình và phương sai là phép rút gọn phân rã được: mỗi nút gửi một bộ ba số cỡ cố định, và phép
gộp cho đúng kết quả của cả tập. Trung vị thì không: muốn trung vị đúng phải gửi nhiều hơn một bộ tóm
tắt cỡ cố định, hoặc chấp nhận một xấp xỉ. Phép nối (join) cũng vậy, thường cần xáo trộn (shuffle) dữ
liệu giữa các nút.

Phần còn lại là thuế phối hợp (coordination tax). Chương nói thao tác cục bộ tính bằng micro giây,
còn phối hợp qua mạng tính bằng mili giây. Định luật Amdahl (phương trình 4) cho biết phần nối tiếp
chặn tốc độ tới mức nào:

$$\text{Speedup} \le \frac{1}{f_\text{serial} + f_\text{parallel} / N_\text{workers}}.$$

In [ ]:
theo_sach("mili giây so với micro giây", 1e-3 / 1e-6, sach=1000, trich="1,000\\(\\times\\) latency difference")
F_NOI_TIEP = sach(0.10, trich="f_{\\text{serial}} = 0.10")
theo_sach("tăng tốc tối đa khi N → ∞", 1 / F_NOI_TIEP, sach=10, trich="cannot exceed \\(10\\times\\)")


def amdahl(f, n):
    return 1 / (f + (1 - f) / n)


N = np.logspace(0, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
for f, mau in zip((0.01, 0.05, F_NOI_TIEP, 0.25), MAU):
    ax.semilogx(N, amdahl(f, N), color=mau, lw=2.5 if f == F_NOI_TIEP else 1.5,
                label=f"f_serial = {f:.2f}" + (" (chương)" if f == F_NOI_TIEP else " (thêm vào để so)"))
ax.axhline(1 / F_NOI_TIEP, color=MAU[2], lw=1, ls=":")
ax.text(1.1, 1 / F_NOI_TIEP + 1.5, "trần 10 lần", fontsize=8, color="0.25")
ax.plot(100, amdahl(F_NOI_TIEP, 100), "o", color="0.15", ms=7, mec="white", mew=1.2)
ax.annotate(f"100 nút: {amdahl(F_NOI_TIEP, 100):.1f} lần", (100, amdahl(F_NOI_TIEP, 100)), xytext=(220, 13.5),
            fontsize=8, va="center", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.7))
ax.set_xlabel("số công nhân N (thang log)")
ax.set_ylabel("tăng tốc tối đa")
ax.set_title("Định luật Amdahl: phần nối tiếp đặt trần")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Với 10 % nối tiếp, 100 nút cho chưa tới 10 lần, và nút thứ 1,000 gần như không thêm gì. Chương rút ra
rằng gỡ phần vào ra nối tiếp được ưu tiên hơn thêm nút tính toán. Chương cũng nhắc rằng một máy chủ với
256 GB RAM xử lý được tập vài terabyte bằng xử lý ngoài bộ nhớ (out-of-core), đọc dần từ đĩa. Trước khi
dựng một cụm phân tán, hãy chắc rằng một máy được viết cẩn thận thật sự không đủ.

## 7. Phiên bản dữ liệu: chép nguyên bản hay lưu theo nội dung

Quản lý phiên bản dữ liệu (data versioning) tốn bao nhiêu chỗ? Chương nêu hai phép tính: mười thí nghiệm trên một tập 100 GB, nếu mỗi thí nghiệm chép nguyên tập,
cần 1 TB; và mười phiên bản của một mô hình 10 GB cần 100 GB. Các công cụ như DVC thay vào đó lưu
theo nội dung: mỗi khối dữ liệu được đặt tên bằng mã băm của chính nó, nên khối nào giống hệt chỉ
lưu một lần, và một phiên bản chỉ là danh sách mã băm.

In [ ]:
SO_TN = sach(10, nguon="100 GB × 10 thí nghiệm = 1 TB bản sao")
TAP_TN = sach(100, nguon="100 GB × 10 thí nghiệm = 1 TB bản sao")
theo_sach("10 thí nghiệm × 100 GB (TB)", SO_TN * TAP_TN / 1000, sach=1, nguon="100 GB × 10 thí nghiệm = 1 TB bản sao")
MO_HINH_GB = sach(10, trich="a 10 GB model naively")
theo_sach("10 phiên bản × mô hình 10 GB (GB)", 10 * MO_HINH_GB, sach=100, trich="would consume 100 GB")
THAM_SO = sach(175e9, nguon="GPT-3 175B tham số")
theo_sach("GPT-3 ở FP32 (GB)", THAM_SO * 4 / 1e9, sach=700, nguon="~700 GB FP32", tol=1)
theo_sach("GPT-3 ở FP16 (GB)", THAM_SO * 2 / 1e9, sach=350, trich="FP16 (350 GB)")
tang = THAM_SO / sach(60e6, trich="AlexNet’s 60M parameters")
theo_sach("AlexNet tới GPT-3, số tham số tăng", tang, sach=2916, trich="2916×", tol=1)
print(f"chính xác là {tang:,.2f} lần: chương cắt phần lẻ, không làm tròn lên 2,917")

Lưu theo nội dung tiết kiệm được bao nhiêu thì tuỳ thay đổi rơi vào đâu. Sổ tay dựng một tập 100,000
dòng × 8 cột, cắt theo hàng thành 100 khối 1,000 dòng, rồi chạy 10 thí nghiệm, mỗi thí nghiệm rẽ ra từ
bản gốc:

* thí nghiệm 1 đến 8 làm sạch vài dòng nằm trong 2 khối;
* thí nghiệm 9 thêm 5,000 dòng mới ở cuối;
* thí nghiệm 10 chuẩn hoá lại một cột, nên chạm vào mọi dòng.

Rồi làm lại với cách cắt theo cột: mỗi khối là một cột của 10,000 dòng.

**Dự đoán:** thí nghiệm 10 tốn bao nhiêu khối mới trong mỗi cách cắt?

In [ ]:
rng = np.random.default_rng(6)
GOC = rng.normal(size=(100_000, 8)).astype(np.float32)


def thi_nghiem(k):
    v = GOC.copy()
    if k <= 8:                                   # làm sạch vài dòng trong 2 khối
        for khoi in rng.choice(100, 2, replace=False):
            v[khoi * 1000 + rng.integers(0, 1000, 5), 0] = 0.0
    elif k == 9:                                 # thêm dòng mới ở cuối
        v = np.vstack([v, rng.normal(size=(5_000, 8)).astype(np.float32)])
    else:                                        # chuẩn hoá lại cột 3
        v[:, 3] = (v[:, 3] - v[:, 3].mean()) / v[:, 3].std()
    return v


def khoi_theo_hang(v):
    return [v[i:i + 1000].tobytes() for i in range(0, len(v), 1000)]


def khoi_theo_cot(v):
    return [np.ascontiguousarray(v[i:i + 10_000, c]).tobytes() for c in range(v.shape[1]) for i in range(0, len(v), 10_000)]


PHIEN_BAN = [GOC] + [thi_nghiem(k) for k in range(1, 11)]
DONG = []
for cat, ham in (("cắt theo hàng", khoi_theo_hang), ("cắt theo cột", khoi_theo_cot)):
    kho, chep, luu = {}, [], []
    for v in PHIEN_BAN:
        for b in ham(v):
            kho.setdefault(hashlib.sha256(b).hexdigest(), len(b))
        chep.append(sum(x.nbytes for x in PHIEN_BAN[:len(chep) + 1]) / 1e6)
        luu.append(sum(kho.values()) / 1e6)
    cua_goc = {hashlib.sha256(x).hexdigest() for x in ham(GOC)}
    moi_10 = sum(1 for b in ham(PHIEN_BAN[-1]) if hashlib.sha256(b).hexdigest() not in cua_goc)
    print(f"{cat}: chép nguyên bản {chep[-1]:.1f} MB, lưu theo nội dung {luu[-1]:.1f} MB "
          f"({chep[-1] / luu[-1]:.1f} lần ít hơn); thí nghiệm 10 tạo {moi_10} khối mới trên {len(ham(GOC))}")
    DONG.append((cat, chep, luu))
assert DONG[0][2][-1] < DONG[0][1][-1] and DONG[1][2][-1] < DONG[0][2][-1]

fig, ax = plt.subplots(figsize=(8, 3.9))
x = np.arange(len(PHIEN_BAN))
ax.plot(x, DONG[0][1], "o-", color="0.45", lw=2, ms=5, label="chép nguyên bản mỗi phiên bản")
for (cat, _, luu), mau in zip(DONG, MAU):
    ax.plot(x, luu, "o-", color=mau, lw=2, ms=5, label=f"lưu theo nội dung, {cat}")
ax.set_xticks(x, ["gốc"] + [str(k) for k in range(1, 11)])
ax.set_xlabel("sau thí nghiệm")
ax.set_ylabel("MB đã lưu, cộng dồn")
ax.set_title("Mười thí nghiệm trên một tập 3.2 MB")
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Tám thí nghiệm làm sạch gần như không tốn thêm gì, vì chỉ vài khối đổi. Thêm dòng ở cuối cũng rẻ.
Nhưng một thay đổi chạm mọi dòng làm mọi khối theo hàng đổi mã băm, và phiên bản đó tốn trọn một bản
sao. Cắt theo cột thì chỉ khối của cột bị sửa đổi. Điểm chương nhấn mạnh với mô hình còn rõ hơn: tinh
chỉnh mọi trọng số tạo ra một checkpoint mà mọi khối đều mới, nên loại bỏ trùng lặp (deduplication) không giúp gì, trừ khi
kho lưu có nén chênh lệch (delta) riêng. Lưu theo nội dung còn cho một thứ nữa miễn phí: danh sách
mã băm của một phiên bản chính là định danh của nó, nên phả hệ dữ liệu (data lineage) chỉ cần ghi
"mô hình này học từ danh sách mã băm này".

## Thử thêm

1. Ở phần 1, thêm một phép lật ngang vào `tang_cuong` cho kho lớn và đo lại. Chi phí tăng bao nhiêu,
   và vì sao phép này hợp với ảnh tự nhiên mà không hợp với chữ số?
2. Ở phần 4, đổi `CHON` thành 50 cột. Thời gian đọc Parquet tăng theo số cột chọn, còn CSV thì sao?
3. Ở phần 6, cho mọi phân vùng cùng kích thước (`co = np.full(SO_NUT, 10_000)`). Trung bình của các
   trung bình còn lệch không?

## Tóm lại

* Giải mã, tăng cường và chuẩn hoá đều tăng theo số điểm ảnh, nên với ảnh lớn, phần chuẩn bị trên CPU
  dễ trở thành điểm nghẽn của huấn luyện.
* Số công nhân cần là một phép khớp năng lực: với 375 ảnh mỗi giây mỗi công nhân và 3,000 ảnh mỗi giây
  ở bộ tăng tốc là 8; với CPU của bạn là con số đo được ở trên.
* Nạp trước giấu khâu ngắn hơn sau khâu dài hơn, nên thời gian mỗi batch nằm giữa max và tổng của hai khâu.
* Đọc 20 trong 100 cột, định dạng theo hàng có η = 0.2 còn định dạng theo cột gần 1, nên trong kịch bản
  chỉ chọn cột của chương, đổi sang Parquet bằng một đường dữ liệu nhanh gấp 5 lần; gzip đổi tốc độ giải
  nén lấy tỉ lệ nén, Snappy làm ngược lại.
* Phép rút gọn phân rã được (tổng và số đếm) tổng hợp ngay tại dữ liệu, nhanh hơn gom về một nút từ
  hàng trăm tới vài nghìn lần; phần nối tiếp 10 % vẫn chặn tăng tốc ở 10 lần.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật dữ liệu](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch04-data-engineering/), dựng từ chương
[*Data Engineering*](https://mlsysbook.ai/vol1/data_engineering/data_engineering.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_04_data_engr.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.